In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
batch_relax_with_M3GNet_UP.py
在给定文件夹下批量用 M3GNet-UP 进行几何弛豫并输出能量
"""

import os, glob, csv, warnings
from pathlib import Path
from pymatgen.core import Structure
from matgl.ext.ase import Relaxer
import matgl                                  # ⬅ 加载模型用
from pymatgen.io.ase import AseAtomsAdaptor

os.environ["MATGL_BACKEND"] = "PYG"  # 关键：用 PyG backend

# 可选并行
# from joblib import Parallel, delayed

warnings.filterwarnings("ignore")  # 清屏

# ------------ 用户参数 -------------
INPUT_DIR   = "1Y_unique"          # 你的掺杂模型文件夹
OUT_DIR     = Path("relaxed")      # 弛豫后 CIF 输出目录
FMAX        = 0.03                 # 收敛阈值 (eV/Å)
MAX_STEPS   = 500                  # 最多优化步
OPTIMIZER   = "FIRE"               # 也可 "BFGS" 等
CSV_NAME    = f"{INPUT_DIR}_relaxed_energies.csv"
# ----------------------------------

OUT_DIR.mkdir(exist_ok=True)

# 1. 载入预训练 M3GNet-UP 势能
potential = matgl.load_model("TensorNet-MatPES-PBE-v2025.1-PES")
relaxer   = Relaxer(potential=potential, relax_cell=True, optimizer=OPTIMIZER)

def relax_one(cif_path: str):
    """弛豫单个结构并返回 (name, natoms, energy, outfile)"""
    struct = Structure.from_file(cif_path)
    result = relaxer.relax(struct, fmax=FMAX, steps=MAX_STEPS)
    final_struct = result["final_structure"]
    energy = float(result["trajectory"].energies[-1])       # eV:contentReference[oaicite:6]{index=6}
    out_file = OUT_DIR / (Path(cif_path).stem + "_relaxed.cif")
    final_struct.to(fmt="cif", filename=str(out_file))
    return Path(cif_path).name, len(final_struct), energy, out_file

# 2. 收集待处理文件
cif_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.cif")))
print(f"[INFO] 待弛豫结构数 = {len(cif_files)}")

# 3. 逐个（或并行）弛豫
results = []
for path in cif_files:
    try:
        results.append(relax_one(path))
        print(f"  ✓ {Path(path).name} 完成")
    except Exception as e:
        print(f"  ✗ {Path(path).name} 失败：{e}")

# 并行版本示例（注释掉，如果想改成并行可取消）
# from joblib import Parallel, delayed
# results = Parallel(n_jobs=4)(delayed(relax_one)(p) for p in cif_files)

# 4. 写 CSV
with open(CSV_NAME, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["file", "natoms", "energy_eV", "relaxed_cif"])
    writer.writerows(results)

print(f"[DONE] 弛豫完毕，结果写入 {CSV_NAME}")


Traceback (most recent call last):
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\matgl\utils\io.py", line 244, in load_model
    return cls_.load(fpaths, **kwargs)
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\matgl\utils\io.py", line 157, in load
    mod = __import__(modname, globals(), locals(), [classname], 0)
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\matgl\models\__init__.py", line 10, in <module>
    from ._chgnet import CHGNet
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\matgl\models\_chgnet.py", line 19, in <module>
    from dgl import readout_edges, readout_nodes
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\dgl\__init__.py", line 16, in <module>
    from . import (
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\dgl\dataloading\__init__.py", line 13, in <module>
    from .dataloader import *
  File "g:\Software\anaconda3\envs\m3gnet\lib\site-packages\dgl\dataloading\dataloader.py", line 27, 

RuntimeError: Unknown error occurred while loading model. Please review the traceback for more information.

In [ ]:
import matgl
matgl.clear_cache()

In [ ]:
import matgl

# 查看可用的预训练模型
print(matgl.get_available_pretrained_models())

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
batch_relax_with_M3GNet_UP.py
在给定文件夹下批量用 M3GNet-UP 进行几何弛豫并输出能量
"""

import os, glob, csv, warnings
from pathlib import Path
from pymatgen.core import Structure
from matgl.ext.ase import Relaxer
import matgl                                  # ⬅ 加载模型用
from pymatgen.io.ase import AseAtomsAdaptor
# 可选并行
# from joblib import Parallel, delayed

warnings.filterwarnings("ignore")  # 清屏

# ------------ 用户参数 -------------
INPUT_DIR   = "2Y_unique"          # 你的掺杂模型文件夹
OUT_DIR     = Path("relaxed")      # 弛豫后 CIF 输出目录
FMAX        = 0.03                 # 收敛阈值 (eV/Å)
MAX_STEPS   = 500                  # 最多优化步
OPTIMIZER   = "FIRE"               # 也可 "BFGS" 等
CSV_NAME    = f"{INPUT_DIR}_relaxed_energies.csv"
# ----------------------------------

OUT_DIR.mkdir(exist_ok=True)

# 1. 载入预训练 M3GNet-UP 势能
potential = matgl.load_model("M3GNet-MP-2021.2.8-PES")      # 官方通用势:contentReference[oaicite:4]{index=4}
relaxer   = Relaxer(potential=potential,
                    relax_cell=True,           # 晶格也一起放
                    optimizer=OPTIMIZER)       # 默认 FIRE:contentReference[oaicite:5]{index=5}

def relax_one(cif_path: str):
    """弛豫单个结构并返回 (name, natoms, energy, outfile)"""
    struct = Structure.from_file(cif_path)
    result = relaxer.relax(struct, fmax=FMAX, steps=MAX_STEPS)
    final_struct = result["final_structure"]
    energy = float(result["trajectory"].energies[-1])       # eV:contentReference[oaicite:6]{index=6}
    out_file = OUT_DIR / (Path(cif_path).stem + "_relaxed.cif")
    final_struct.to(fmt="cif", filename=str(out_file))
    return Path(cif_path).name, len(final_struct), energy, out_file

# 2. 收集待处理文件
cif_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.cif")))
print(f"[INFO] 待弛豫结构数 = {len(cif_files)}")

# 3. 逐个（或并行）弛豫
results = []
for path in cif_files:
    try:
        results.append(relax_one(path))
        print(f"  ✓ {Path(path).name} 完成")
    except Exception as e:
        print(f"  ✗ {Path(path).name} 失败：{e}")

# 并行版本示例（注释掉，如果想改成并行可取消）
# from joblib import Parallel, delayed
# results = Parallel(n_jobs=4)(delayed(relax_one)(p) for p in cif_files)

# 4. 写 CSV
with open(CSV_NAME, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["file", "natoms", "energy_eV", "relaxed_cif"])
    writer.writerows(results)

print(f"[DONE] 弛豫完毕，结果写入 {CSV_NAME}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
batch_relax_with_M3GNet_UP.py
在给定文件夹下批量用 M3GNet-UP 进行几何弛豫并输出能量
"""

import os, glob, csv, warnings
from pathlib import Path
from pymatgen.core import Structure
from matgl.ext.ase import Relaxer
import matgl                                  # ⬅ 加载模型用
from pymatgen.io.ase import AseAtomsAdaptor
# 可选并行
# from joblib import Parallel, delayed

warnings.filterwarnings("ignore")  # 清屏

# ------------ 用户参数 -------------
INPUT_DIR   = "3Y_unique"          # 你的掺杂模型文件夹
OUT_DIR     = Path("relaxed")      # 弛豫后 CIF 输出目录
FMAX        = 0.03                 # 收敛阈值 (eV/Å)
MAX_STEPS   = 500                  # 最多优化步
OPTIMIZER   = "FIRE"               # 也可 "BFGS" 等
CSV_NAME    = f"{INPUT_DIR}_relaxed_energies.csv"
# ----------------------------------

OUT_DIR.mkdir(exist_ok=True)

# 1. 载入预训练 M3GNet-UP 势能
potential = matgl.load_model("M3GNet-MP-2021.2.8-PES")      # 官方通用势:contentReference[oaicite:4]{index=4}
relaxer   = Relaxer(potential=potential,
                    relax_cell=True,           # 晶格也一起放
                    optimizer=OPTIMIZER)       # 默认 FIRE:contentReference[oaicite:5]{index=5}

def relax_one(cif_path: str):
    """弛豫单个结构并返回 (name, natoms, energy, outfile)"""
    struct = Structure.from_file(cif_path)
    result = relaxer.relax(struct, fmax=FMAX, steps=MAX_STEPS)
    final_struct = result["final_structure"]
    energy = float(result["trajectory"].energies[-1])       # eV:contentReference[oaicite:6]{index=6}
    out_file = OUT_DIR / (Path(cif_path).stem + "_relaxed.cif")
    final_struct.to(fmt="cif", filename=str(out_file))
    return Path(cif_path).name, len(final_struct), energy, out_file

# 2. 收集待处理文件
cif_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.cif")))
print(f"[INFO] 待弛豫结构数 = {len(cif_files)}")

# 3. 逐个（或并行）弛豫
results = []
for path in cif_files:
    try:
        results.append(relax_one(path))
        print(f"  ✓ {Path(path).name} 完成")
    except Exception as e:
        print(f"  ✗ {Path(path).name} 失败：{e}")

# 并行版本示例（注释掉，如果想改成并行可取消）
# from joblib import Parallel, delayed
# results = Parallel(n_jobs=4)(delayed(relax_one)(p) for p in cif_files)

# 4. 写 CSV
with open(CSV_NAME, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["file", "natoms", "energy_eV", "relaxed_cif"])
    writer.writerows(results)

print(f"[DONE] 弛豫完毕，结果写入 {CSV_NAME}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
batch_relax_with_M3GNet_UP.py
在给定文件夹下批量用 M3GNet-UP 进行几何弛豫并输出能量
"""

import os, glob, csv, warnings
from pathlib import Path
from pymatgen.core import Structure
from matgl.ext.ase import Relaxer
import matgl                                  # ⬅ 加载模型用
from pymatgen.io.ase import AseAtomsAdaptor
# 可选并行
# from joblib import Parallel, delayed

warnings.filterwarnings("ignore")  # 清屏

# ------------ 用户参数 -------------
INPUT_DIR   = "4Y_unique"          # 你的掺杂模型文件夹
OUT_DIR     = Path("relaxed")      # 弛豫后 CIF 输出目录
FMAX        = 0.03                 # 收敛阈值 (eV/Å)
MAX_STEPS   = 500                  # 最多优化步
OPTIMIZER   = "FIRE"               # 也可 "BFGS" 等
CSV_NAME    = f"{INPUT_DIR}_relaxed_energies.csv"
# ----------------------------------

OUT_DIR.mkdir(exist_ok=True)

# 1. 载入预训练 M3GNet-UP 势能
potential = matgl.load_model("M3GNet-MP-2021.2.8-PES")      # 官方通用势:contentReference[oaicite:4]{index=4}
relaxer   = Relaxer(potential=potential,
                    relax_cell=True,           # 晶格也一起放
                    optimizer=OPTIMIZER)       # 默认 FIRE:contentReference[oaicite:5]{index=5}

def relax_one(cif_path: str):
    """弛豫单个结构并返回 (name, natoms, energy, outfile)"""
    struct = Structure.from_file(cif_path)
    result = relaxer.relax(struct, fmax=FMAX, steps=MAX_STEPS)
    final_struct = result["final_structure"]
    energy = float(result["trajectory"].energies[-1])       # eV:contentReference[oaicite:6]{index=6}
    out_file = OUT_DIR / (Path(cif_path).stem + "_relaxed.cif")
    final_struct.to(fmt="cif", filename=str(out_file))
    return Path(cif_path).name, len(final_struct), energy, out_file

# 2. 收集待处理文件
cif_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.cif")))
print(f"[INFO] 待弛豫结构数 = {len(cif_files)}")

# 3. 逐个（或并行）弛豫
results = []
for path in cif_files:
    try:
        results.append(relax_one(path))
        print(f"  ✓ {Path(path).name} 完成")
    except Exception as e:
        print(f"  ✗ {Path(path).name} 失败：{e}")

# 并行版本示例（注释掉，如果想改成并行可取消）
# from joblib import Parallel, delayed
# results = Parallel(n_jobs=4)(delayed(relax_one)(p) for p in cif_files)

# 4. 写 CSV
with open(CSV_NAME, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["file", "natoms", "energy_eV", "relaxed_cif"])
    writer.writerows(results)

print(f"[DONE] 弛豫完毕，结果写入 {CSV_NAME}")
